# Tutorial 11-4: Practical Association Rule Mining on the Online Retail Dataset
**Course:** CSEN 140: Data Mining/Machine Learning  
**Instructor:** Dr. David C. Anastasiu

--- 
## Objective
This tutorial applies the concepts from Lectures 11-1 through 11-3 to a real-world dataset: the **Online Retail** dataset. We will explore how to clean transactional data, transform it for mining, and extract actionable rules while identifying compact pattern representations.

### Key Tasks
1. **Data Preprocessing:** Standardizing raw retail logs.
2. **Frequent Itemset Generation:** Using Apriori to find patterns that meet the minimum support threshold (minsup).
3. **Rule Extraction:** Generating rules based on confidence (minconf).
4. **Compact Filtering:** Distinguishing between Frequent, Closed, and Maximal itemsets.

In [ ]:
!python -m pip install mlxtend
!python -m pip install --upgrade pandas openpyxl

import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Load the real-world dataset
# We use the UCI Online Retail dataset (Excel format)
url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'
df = pd.read_excel(url)

# 2. Clean the Data
df['Description'] = df['Description'].str.strip()
df.dropna(axis=0, subset=['InvoiceNo'], inplace=True)
df['InvoiceNo'] = df['InvoiceNo'].astype('str')
df = df[~df['InvoiceNo'].str.contains('C')] # Remove cancellations 

# Filter to a specific country to manage complexity
df_france = df[df['Country'] == 'France']

# Create the basket using pivot_table (more robust than groupby/unstack in some environments)
basket = pd.pivot_table(df_france, values='Quantity', index='InvoiceNo', 
                        columns='Description', aggfunc='sum').fillna(0)

# 3. One-Hot Encoding (Binary Representation)
def encode_units(x):
    return 1 if x >= 1 else 0

basket_sets = basket.map(encode_units)
print(f"Dataset shape: {basket_sets.shape}")
basket_sets.head()

### 1. Frequent Itemset Generation (minsup)
We find itemsets with a support threshold ($\ge$ minsup). Given the sparsity of retail data, we'll start with 7%.

In [ ]:
frequent_itemsets = apriori(basket_sets, min_support=0.07, use_colnames=True)
frequent_itemsets.sort_values(by='support', ascending=False).head(10)

### 2. Rule Generation (minconf)
Next, we generate high confidence rules ($\ge$ minconf) from each frequent itemset.

In [ ]:
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.8)
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(by='lift', ascending=False).head(10)

### 3. Identifying Maximal and Closed Itemsets
Finally, we apply our definitions for compact representations to filter out redundancies.

In [ ]:
# Find Maximal Frequent Itemsets (no immediate superset is frequent)
is_maximal = []
for i, row in frequent_itemsets.iterrows():
    is_superset = False
    for j, other_row in frequent_itemsets.iterrows():
        if i != j and row['itemsets'].issubset(other_row['itemsets']):
            is_superset = True
            break
    is_maximal.append(not is_superset)
maximal_itemsets = frequent_itemsets[is_maximal]

# Find Closed Frequent Itemsets (no immediate superset has same support)
is_closed = []
for i, row in frequent_itemsets.iterrows():
    is_superset_same_support = False
    for j, other_row in frequent_itemsets.iterrows():
        if i != j and row['itemsets'].issubset(other_row['itemsets']) and row['support'] == other_row['support']:
            is_superset_same_support = True
            break
    is_closed.append(not is_superset_same_support)
closed_itemsets = frequent_itemsets[is_closed]

print(f"Total Frequent Itemsets: {len(frequent_itemsets)}")
print(f"Closed Frequent Itemsets: {len(closed_itemsets)}")
print(f"Maximal Frequent Itemsets: {len(maximal_itemsets)}")

---
## Summary & Conclusion

This practical demo confirms the core principles:
1. **Preprocessing is Key:** Real-world transactions must be cleaned and encoded before mining.
2. **Decoupling Requirements:** Finding frequent itemsets and generating rules are distinct steps that allow for computational efficiency.
3. **Compression Matters:** Closed and Maximal itemsets provide a manageable summary of the pattern lattice.